In [1]:
# PHASE 8 — Dashboard Data Prep
import pandas as pd
import numpy as np

risk = pd.read_csv("../data/phase7_risk_scores.csv")

detail_cols = ['customer_id', 'age', 'gender', 'location', 'income_bracket',
               'customer_segment', 'acquisition_channel',
               'days_since_last_tnx', 'avg_amount_change_pct', 'withdrawal_share_shift',
               'failed_transactions', 'support_tickets_count', 'satisfaction_score',
               'app_logins_frequency']
details = pd.read_csv("../data/phase6_modeling_data.csv", usecols=detail_cols)

customers = risk.merge(details, on='customer_id', how='left', validate='one_to_one')

# Easier-to-read columns for the dashboards
customers['city'] = customers['location'].str.split(',').str[0]          # "Bogotá, Cundinamarca" → "Bogotá"
customers['age_group'] = pd.cut(customers['age'], bins=[0, 34, 44, 54, 120],
                                labels=['Under 35', '35-44', '45-54', '55+'])
customers = customers.drop(columns=['location'])

print(customers.shape)
print("Missing values:", customers.isna().sum().sum())


(48723, 24)
Missing values: 0


In [2]:
amt = customers['avg_amount_change_pct']
wds = customers['withdrawal_share_shift']
days = customers['days_since_last_tnx']

# Checked in order: strongest signal first
conditions = [
    amt <= -0.35,                     # transaction amounts dropped 35%+
    (wds <= -0.15) | (wds >= 0.25),   # withdrawal share shifted sharply either way
    amt >= 0.44,                      # transaction amounts surged 44%+
    days >= 14,                       # no transaction in 14+ days
]
signals = [
    'Sharp drop in transaction amounts',
    'Sharp shift in withdrawal share',
    'Sharp surge in transaction amounts',
    'Inactive 14+ days',
]
customers['primary_warning_signal'] = np.select(conditions, signals, default='No clear signal')

# Validate: does each signal really carry higher churn?
customers.groupby('primary_warning_signal').agg(
    customers=('customer_id', 'count'),
    actual_churn_rate=('churned', 'mean'),
    avg_churn_probability=('churn_probability', 'mean'),
).sort_values('actual_churn_rate', ascending=False).round(3)


,customers,actual_churn_rate,avg_churn_probability
primary_warning_signal,,,
Sharp drop in transaction amounts,9578,0.381,0.357
Sharp shift in withdrawal share,11019,0.281,0.267
Sharp surge in transaction amounts,3065,0.232,0.223
Inactive 14+ days,8293,0.138,0.160
No clear signal,16768,0.100,0.125


In [5]:
tx = pd.read_csv("../data/transactions_data.csv", parse_dates=['date'])
tx['month'] = tx['date'].dt.to_period('M').dt.to_timestamp()      # e.g. 2023-03-17 → 2023-03-01

# Attach each transaction's customer group
tx = tx.merge(risk[['customer_id', 'churned', 'risk_category']], on='customer_id', how='inner')

monthly = tx.groupby(['month', 'churned', 'risk_category']).agg(
    active_customers=('customer_id', 'nunique'),
    transactions=('amount', 'size'),
    total_amount=('amount', 'sum'),
).reset_index()

# Group sizes, so the dashboards can calculate "% of the group active this month"
group_size = risk.groupby(['churned', 'risk_category']).size().rename('group_customers').reset_index()
monthly = monthly.merge(group_size, on=['churned', 'risk_category'])

print(monthly.shape)
monthly.head(10
             )


(72, 7)


,month,churned,risk_category,active_customers,transactions,total_amount,group_customers
0,2023-01-01,False,High (> 0.30),3205,6162,26118441300,4624
1,2023-01-01,False,Low (<0.20),19265,203075,785597337200,23366
2,2023-01-01,False,Medium (0.20-0.30),7719,19384,62320553600,10456
3,2023-01-01,True,High (> 0.30),3159,5851,21183463600,4262
4,2023-01-01,True,Low (<0.20),2191,5763,19147407200,2689
5,2023-01-01,True,Medium (0.20-0.30),2600,5406,17519606250,3326
6,2023-02-01,False,High (> 0.30),3159,6104,23860873900,4624
7,2023-02-01,False,Low (<0.20),19217,202530,767913380750,23366
8,2023-02-01,False,Medium (0.20-0.30),7673,19259,60588888050,10456
9,2023-02-01,True,High (> 0.30),3130,5809,20228557400,4262


In [4]:
customers.to_csv("../data/phase8_dashboard_customers.csv", index=False)
monthly.to_csv("../data/phase8_dashboard_monthly.csv", index=False)
print("Saved:", customers.shape, monthly.shape)


Saved: (48723, 25) (72, 7)
